In [ ]:
# =========================
# 0) (Local) Install deps from requirements.txt ONLY
# =========================
from pathlib import Path
import sys, os, glob, subprocess

ROOT = Path().resolve()

# Find requirements.txt in project root or nested (e.g., exports)
reqs = []
if (ROOT / "requirements.txt").exists():
    reqs = [ROOT / "requirements.txt"]
else:
    reqs = [Path(p) for p in glob.glob("**/requirements.txt", recursive=True)]

if not reqs:
    raise FileNotFoundError("No requirements.txt found. Please add one with all dependencies.")

# Prefer the shortest path (closest to root)
req_file = sorted(reqs, key=lambda p: len(str(p)))[0]
print("Installing from:", req_file)

# Single install command — no extra pip installs anywhere else
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file), "--no-cache-dir"])
print("Requirements installed.")

In [ ]:
# =========================
# Bootstrapping imports
# =========================
from pathlib import Path
import sys, os, re
import pandas as pd
from dotenv import load_dotenv, find_dotenv

ROOT = Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

In [ ]:
# =========================
# Module imports (from file 1)
# =========================
from utils.io import answers_to_csv
from pipeline.evaluator import evaluate
from pipeline.router import route_question
from pipeline.agent import solve_with_router

In [ ]:
# =========================
# Env (tokens & model IDs)
# =========================
load_dotenv(find_dotenv(), override=False)    # safe no-op if .env missing

# If your HF key is set as HF_TOKEN locally, mirror it
os.environ["HF_API_KEY"] = os.environ.get("HF_API_KEY", os.environ.get("HF_TOKEN", ""))

# Router = local 7B on GPU (LLM-first router)
os.environ.setdefault("ROUTER_MODEL_ID", "Qwen/Qwen2.5-Math-7B-Instruct")

# Solver default = remote 235B (via your existing client)
os.environ.setdefault("FALLBACK_MODEL_ID", "Qwen/Qwen3-235B-A22B-Thinking-2507")

# Use local GPU for router; solver remains your fallback unless you change it
os.environ.setdefault("HF_BACKEND", "local")   # ensures router uses local transformers GPU
os.environ.setdefault("LOCAL_SOLVER", "0")     # keep "0" unless you added a local-solver shim

In [ ]:
# =========================
# Load prompt from file
# =========================
USER_PROMPT = Path("data/paper.txt").read_text(encoding="utf-8")

In [ ]:
# =========================
# Run solver → predictions CSV
# =========================
answers, qtexts = solve_with_router(
    USER_PROMPT,
    batch_size=20,
    # model_map={}  # (keep for future multi-LLM routing; single-LLM now)
    use_langchain_router=False,         # local HF router (no OpenAI)
    prefer_regex_on_conflict=True,      # regex corrects LLM disagreements
    temperature=0.0,
    max_tokens=20000,
    verbose=True,
)

pred_csv = answers_to_csv(answers, qtexts, out_path="data/model_preds.csv")
print("Saved predictions:", pred_csv)

In [ ]:
# =========================
# Evaluate vs answer_key.csv
# =========================
result = evaluate(
    key_path="data/answer_key.csv",
    pred_path=pred_csv,
    topic_col="class_type",   # set None if your key has no 'class_type'
    out_dir="data"
)
print("== SUMMARY ==")
print(result["summary"])

In [ ]:
# =========================
# Build mismatches CSV (robust: reloads from disk, recomputes qtype if missing)
# =========================
from pathlib import Path
import re
import pandas as pd
from pipeline.router import route_question

# 1) Paths (use pred_csv variable if it exists; else default path)
pred_path = Path(pred_csv) if 'pred_csv' in globals() else Path("data/model_preds.csv")
key_path  = Path("data/answer_key.csv")
paper_path = Path("data/paper.txt")

assert pred_path.exists(), f"Predictions file not found: {pred_path}"
assert key_path.exists(),  f"Answer key not found: {key_path}"
assert paper_path.exists(),f"Paper not found: {paper_path}"

# 2) Load predictions
preds_df = pd.read_csv(pred_path)
preds_df["qid"] = pd.to_numeric(preds_df["qid"], errors="coerce").astype("Int64")

# 3) Ensure qtype column exists (compute on the fly if missing)
if "qtype" not in preds_df.columns or preds_df["qtype"].isna().all():
    USER_PROMPT = paper_path.read_text(encoding="utf-8")
    anchors = list(re.finditer(r"(Q(?P<n>\d+)\s*\.?\s*:?)", USER_PROMPT, flags=re.I))
    rows = []
    for i, m in enumerate(anchors):
        n = int(m.group("n"))
        start = m.start()
        end = anchors[i+1].start() if i+1 < len(anchors) else len(USER_PROMPT)
        chunk = USER_PROMPT[start:end].strip()
        q_text = re.split(r"\(\s*1\s*\)", chunk, maxsplit=1)[0].strip()
        q_text_clean = re.sub(r"\s+", " ", q_text)
        lbl = route_question(
            q_text_clean,
            use_langchain_router=False,     # HF-only routing
            prefer_regex_on_conflict=True
        )
        rows.append({"qid": n, "qtype": lbl})
    qtypes_df = pd.DataFrame(rows)
    qtypes_df["qid"] = pd.to_numeric(qtypes_df["qid"], errors="coerce").astype("Int64")
    preds_df = preds_df.drop(columns=["qtype"], errors="ignore").merge(qtypes_df, on="qid", how="left")
    preds_df.to_csv(pred_path, index=False)
    print("Computed and merged qtype into predictions.")

# 4) Load key & compute correctness
key_df = pd.read_csv(key_path)
key_df["qid"] = pd.to_numeric(key_df["qid"], errors="coerce").astype("Int64")

df = preds_df.merge(
    key_df[["qid","option_index"]].rename(columns={"option_index":"answer"}),
    on="qid", how="left",
    validate="m:1"
)
df["correct"] = (df["option_index"] == df["answer"])

# 5) Save mismatches (single file)
mism = df[~df["correct"]].sort_values("qid")
mism_path = Path("data/model_preds_mismatch.csv")
cols = [c for c in ["qid","qtype","option_index","answer","explanation"] if c in mism.columns]
mism[cols].to_csv(mism_path, index=False)
print("Saved mismatches to:", mism_path)

# (Optional) quick per-type accuracy
try:
    print("\nPer-type accuracy:")
    print(df.groupby("qtype", dropna=False)["correct"].mean().sort_values(ascending=False))
except Exception as e:
    print("Per-type breakdown skipped:", e)

In [ ]:
# =========================
# Fix NaN qtype in model_preds.csv (robust)
# =========================
from pathlib import Path
import re
import pandas as pd
from pipeline.router import route_question
from pipeline.planner import parse_questions  # already in your project

# Prefer the ROOT variable from earlier cells; fallback to current folder
ROOT = Path(globals().get("ROOT", Path().resolve()))

pred_path  = ROOT / "data" / "model_preds.csv"
paper_path = ROOT / "data" / "paper.txt"
assert pred_path.exists(),  f"Predictions file not found: {pred_path}"
assert paper_path.exists(), f"Paper not found: {paper_path}"

# 1) Load predictions and normalize qid dtype
preds_df = pd.read_csv(pred_path)
preds_df["qid"] = pd.to_numeric(preds_df["qid"], errors="coerce").astype("Int64")

# 2) Parse paper and compute qtype for each question id
text = paper_path.read_text(encoding="utf-8")

# ✅ Use the correct regex (no double-escaping inside a raw string)
anchors = list(re.finditer(r"(Q(?P<n>\d+)\s*\.?\s*:?)", text, flags=re.I))
print("Anchors found:", len(anchors))

rows = []
for i, m in enumerate(anchors):
    n = int(m.group("n"))
    start = m.start()
    end   = anchors[i+1].start() if i+1 < len(anchors) else len(text)
    chunk = text[start:end].strip()

    # Split at "(1)" to isolate question stem
    q_text = re.split(r"\(\s*1\s*\)", chunk, maxsplit=1)[0].strip()
    q_text_clean = re.sub(r"\s+", " ", q_text)

    # LLM router on HF (or regex-only if the model isn’t available)
    label = route_question(
        q_text_clean,
        use_langchain_router=False,        # no OpenAI needed
        prefer_regex_on_conflict=True
    )
    rows.append({"qid": n, "qtype": label})

qtypes_df = pd.DataFrame(rows)
qtypes_df["qid"] = pd.to_numeric(qtypes_df["qid"], errors="coerce").astype("Int64")
qtypes_df = qtypes_df.drop_duplicates("qid", keep="first")

# 3) Merge into the SAME predictions file
merged = preds_df.drop(columns=["qtype"], errors="ignore").merge(qtypes_df, on="qid", how="left")

# 4) If any qtype are still NaN, fill by re-parsing with your parser to get exactly the same text
if merged["qtype"].isna().any():
    qtexts, _opts = parse_questions(text)  # {'q1': '...', ...}
    def _fill_lbl(row):
        if pd.notna(row["qtype"]):
            return row["qtype"]
        qid = row["qid"]
        if pd.isna(qid):
            return "unknown"
        qt = qtexts.get(f"q{int(qid)}", "")
        if not qt:
            return "unknown"
        return route_question(qt, use_langchain_router=False, prefer_regex_on_conflict=True)

    merged["qtype"] = merged.apply(_fill_lbl, axis=1)

merged["qtype"] = merged["qtype"].fillna("unknown")

# 5) Save back
merged.to_csv(pred_path, index=False)
print("Wrote qtype into:", pred_path)

print("\nPredictions head (with qtype):")
print(merged.head(8)[["qid","option_index","qtype","explanation"]])
